In [1]:
import os
import platform
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re

COLUNAS = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "TRACTORA", "REMOLQUE",
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT", "PESO_BRUTO",
    "CODEUT", "ESTADO_UT", "RANGO_UT", "FCARGA", "ACTIVIDAD",
    "CODEDT", "ESTADO_DT", "REFERENCIA", "CODACT", "LOCORIGEN",
    "PROV_ORIGEN", "PAISORIGEN", "CPOSTAL", "LOCDESTINO", "PROV_DESTINO",
    "PAISDESTINO", "CPOSTAD", "KM", "FENTREGA", "ORIGEN",
    "ENTREGAR", "PROV_ENTREGAR", "PAISENTREGAR", "DESTINO", "PALETS",
    "PREFAC", "RUTA", "COBROREAL", "GESTION", "DEPART",
    "USCODE", "USUARIO", "TIPOCLIENTE", "TIPOFLUJO", "WMSCODRGT",
    "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TEMP_MERC_PED",
    "TIPOPALETA", "CAMION_TIPO", "CAMION_CAPACIDAD", "TIPO_COMBUSTIBLE",
    "KMREALES", "ALBARAN"
]

COLUNA_DATA = "FENTREGA"
COLUNA_ORIGEM = "ficheiro_origem"
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"
REGEX_ANO = re.compile(r"^(\d{4})")

In [2]:
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27")
elif platform.system() == 'Darwin':
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = Path("inform_27")

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ BD: {DB_PATH}")

✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db


In [3]:
import csv

# ==========================
# Converter XLS → CSV (Streaming, sem carregar tudo em RAM)
# ==========================
def ler_xls_streaming(caminho):
    """Retorna (cabecalho, generator de linhas)."""
    tree = ET.parse(caminho)
    raiz = tree.getroot()
    worksheets = raiz.findall(".//ss:Worksheet", NS)

    if not worksheets:
        raise ValueError("Nenhuma worksheet encontrada.")

    def ler_linha(row):
        valores = []
        proximo = 1
        for cell in row.findall("ss:Cell", NS):
            indice = cell.get(SS_INDEX)
            indice = int(indice) if indice else proximo
            while len(valores) < indice - 1:
                valores.append(None)
            data = cell.find("ss:Data", NS)
            valores.append(data.text if data is not None else None)
            proximo = indice + 1
        return valores

    # Extrair cabeçalho da 1ª sheet
    primeira_tabela = worksheets[0].find("ss:Table", NS)
    primeira_row = primeira_tabela.findall("ss:Row", NS)[0]
    cabecalho = ler_linha(primeira_row)
    cabecalho = [str(v).strip() if v else f"COLUNA_{i + 1}" for i, v in enumerate(cabecalho)]

    def gerar_linhas():
        for num_sheet, worksheet in enumerate(worksheets):
            tabela = worksheet.find("ss:Table", NS)
            if tabela is None:
                continue
            rows = tabela.findall("ss:Row", NS)
            if not rows:
                continue
            linhas_comeco = 1 if num_sheet == 0 else 0
            for row in rows[linhas_comeco:]:
                valores = ler_linha(row)
                if len(valores) < len(cabecalho):
                    valores += [None] * (len(cabecalho) - len(valores))
                yield valores[:len(cabecalho)]

    return cabecalho, gerar_linhas()


ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))

if not ficheiros:
    print(f"⚠️  Nenhum ficheiro .xls em: {PASTA_FICHEIROS}")
else:
    total_convertidos = 0

    for numero, caminho in enumerate(ficheiros, 1):
        try:
            contador = 0
            cabecalho, linhas_gen = ler_xls_streaming(caminho)

            with open(PASTA_FICHEIROS / f"{caminho.stem}.csv", "w", newline="", encoding="utf-8", buffering=8192) as f:
                writer = csv.writer(f)
                writer.writerow(cabecalho)
                for linha in linhas_gen:
                    writer.writerow(linha)
                    contador += 1

            total_convertidos += 1
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} → CSV ({contador:,} linhas)")

        except Exception as erro:
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} — ERRO: {erro}")

    print(f"\n✓ {total_convertidos} ficheiros convertidos")

[1/4] SAL_DAT027_2024_01_07.xls → CSV (800,341 linhas)
[2/4] SAL_DAT027_2024_07_12.xls → CSV (703,177 linhas)
[3/4] SAL_DAT027_2025_01_07.xls → CSV (735,018 linhas)
[4/4] SAL_DAT027_2025_07_12.xls → CSV (668,662 linhas)

✓ 4 ficheiros convertidos


In [4]:
import csv

# ==========================
# Funções auxiliares
# ==========================
def obter_ano(valor):
    if valor is None:
        return None
    texto = str(valor).strip()
    match = REGEX_ANO.match(texto)
    return int(match.group(1)) if match and match.group(1) != "0000" else None


def tabela_ano(ano):
    return f"inform_27_{ano}"


def criar_tabela(con, ano):
    tabela = tabela_ano(ano)
    colunas_sql = ", ".join(f'"{c}" TEXT' for c in COLUNAS)

    con.execute(f'''
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql},
            "{COLUNA_ORIGEM}" TEXT
        )
    ''')

    con.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS "idx_{tabela}" '
        f'ON "{tabela}" ("CODEDT", "FENTREGA", "CODEUT")'
    )
    return tabela


def inserir_linhas(con, ano, linhas):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(con, ano)
    colunas_sql = ", ".join(f'"{c}"' for c in COLUNAS)
    placeholders = ", ".join("?" for _ in range(len(COLUNAS) + 1))

    sql = f'INSERT OR IGNORE INTO "{tabela}" ({colunas_sql}, "{COLUNA_ORIGEM}") VALUES ({placeholders})'

    antes = con.total_changes
    con.executemany(sql, linhas)
    inseridos = con.total_changes - antes

    return inseridos, len(linhas) - inseridos


# ==========================
# 1. Processamento ficheiro a ficheiro
# ==========================
ficheiros = sorted(PASTA_FICHEIROS.rglob("*.csv"))

if not ficheiros:
    raise FileNotFoundError(f"Nenhum ficheiro .csv em: {PASTA_FICHEIROS}")

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA synchronous = OFF")
con.execute("PRAGMA journal_mode = WAL")

totais = {
    "ficheiros": 0,
    "linhas": 0,
    "inseridos": {},
    "duplicados": 0,
    "sem_data": 0,
    "erros": 0
}

for numero, caminho in enumerate(ficheiros, 1):
    try:
        # ==========================
        # 2. Ler ficheiro CSV
        # ==========================
        with open(caminho, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            cabecalho = next(reader)
            linhas = list(reader)

        if COLUNA_DATA not in cabecalho:
            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{caminho.name} — sem {COLUNA_DATA}"
            )
            totais["erros"] += 1
            continue

        mapa_indices = {
            coluna: i
            for i, coluna in enumerate(cabecalho)
        }

        linhas_por_ano = {}
        sem_data = 0

        # ==========================
        # 3. Preparar linhas
        # ==========================
        for valores in linhas:
            indice_data = mapa_indices.get(COLUNA_DATA)
            valor_data = (
                valores[indice_data]
                if indice_data is not None and indice_data < len(valores)
                else None
            )

            ano = obter_ano(valor_data)

            if ano is None:
                sem_data += 1
                continue

            linha = []
            for coluna in COLUNAS:
                indice = mapa_indices.get(coluna)
                valor = (
                    valores[indice]
                    if indice is not None and indice < len(valores)
                    else None
                )
                linha.append(valor)

            linha.append(caminho.name)
            linhas_por_ano.setdefault(ano, []).append(linha)

        # ==========================
        # 4. Inserir dados
        # ==========================
        novos_ficheiro = 0
        duplicados_ficheiro = 0

        for ano, batch in sorted(linhas_por_ano.items()):
            inseridos, duplicados = inserir_linhas(
                con,
                ano,
                batch
            )
            totais["inseridos"][ano] = (
                totais["inseridos"].get(ano, 0) + inseridos
            )
            novos_ficheiro += inseridos
            duplicados_ficheiro += duplicados

        # ==========================
        # 5. Confirmar ficheiro
        # ==========================
        con.commit()

        totais["ficheiros"] += 1
        totais["linhas"] += len(linhas)
        totais["duplicados"] += duplicados_ficheiro
        totais["sem_data"] += sem_data

        print(
            f"[{numero}/{len(ficheiros)}] "
            f"{caminho.name} | "
            f"+{novos_ficheiro:,} novos | "
            f"{duplicados_ficheiro:,} duplicados | "
            f"{sem_data:,} sem data"
        )

    except Exception as erro:
        # ==========================
        # 6. Reverter apenas ficheiro
        # ==========================
        con.rollback()
        totais["erros"] += 1
        print(
            f"[{numero}/{len(ficheiros)}] "
            f"{caminho.name} — ERRO: {erro}"
        )

# ==========================
# 7. Fechar ligação
# ==========================
con.close()

# ==========================
# 8. Resumo final
# ==========================
print("\n--- RESUMO ---")
print(
    f"Ficheiros: {totais['ficheiros']} | "
    f"Linhas lidas: {totais['linhas']:,} | "
    f"Duplicadas: {totais['duplicados']:,} | "
    f"Sem data: {totais['sem_data']:,} | "
    f"Erros: {totais['erros']}"
)
for ano, qtd in sorted(totais["inseridos"].items()):
    print(f"Linhas novas em {ano}: {qtd:,}")

[1/4] SAL_DAT027_2024_01_07.csv | +35 novos | 800,223 duplicados | 83 sem data
[2/4] SAL_DAT027_2024_07_12.csv | +2 novos | 703,088 duplicados | 87 sem data
[3/4] SAL_DAT027_2025_01_07.csv | +664,068 novos | 70,853 duplicados | 97 sem data
[4/4] SAL_DAT027_2025_07_12.csv | +585,475 novos | 83,099 duplicados | 88 sem data

--- RESUMO ---
Ficheiros: 4 | Linhas lidas: 2,907,198 | Duplicadas: 1,657,263 | Sem data: 355 | Erros: 0
Linhas novas em 2023: 35
Linhas novas em 2024: 0
Linhas novas em 2025: 1,249,482
Linhas novas em 2026: 61
Linhas novas em 2027: 2


codigos = [
    3480811, 3481516, 3481696, 3481698, 3481943, 3482677, 3483108, 3483445,
    3483592, 3483736, 3484056, 3484077, 3485200, 3485406, 3485969, 3485992,
    3486113, 3486196, 3486219, 3487230, 3487244, 3488343, 3489063, 3489324,
    3489525, 3489574, 3489582, 3489765, 3490755, 3491022, 3491279, 3491791,
    3491839, 3492300, 3492858, 3492933, 3493024, 3493824, 3493846, 3493986,
    3494242, 3494319, 3494763, 3494785, 3495711, 3495888, 3496028, 3496205,
    3497432, 3497580, 3498348, 3498427, 3498624, 3499392, 3499415, 3499605,
    3500371, 3500452, 3500976, 3500994, 3501539, 3501560, 3502199, 3502561,
    3503104, 3503331, 3503912, 3504136, 3504400, 3504957, 3505156, 3480783,
    3480786, 3480792, 3480799, 3480800, 3480821, 3480831, 3481107, 3481109,
    3481146, 3481159, 3481200, 3481203, 3481222, 3481224, 3481371, 3481373,
    3481377, 3481517, 3481519, 3481521, 3481522, 3481557, 3481561, 3481576,
    3481624, 3481626, 3481627, 3481628, 3481629, 3481630, 3481644, 3481682,
    3481685, 3481687, 3481694, 3481727, 3481884, 3481947, 3481948, 3481996,
    3482005, 3482103, 3482229, 3482340, 3482385, 3482440, 3482487, 3482506,
    3482517, 3482535, 3482537, 3482538, 3482540, 3482541, 3482545, 3482549,
    3482652, 3482658, 3482661, 3482662, 3482663, 3482664, 3482673, 3482675,
    3482683, 3482837, 3482848, 3482868, 3483014, 3483026, 3483028, 3483031,
    3483036, 3483130, 3483145, 3483148, 3483150, 3483151, 3483154, 3483208,
    3483212, 3483220, 3483383, 3483403, 3483424, 3483461, 3483464, 3483523,
    3483526, 3483536, 3483541, 3483552, 3483560, 3483563, 3483569, 3483574,
    3483575, 3483576, 3483577, 3483579, 3483585, 3483697, 3483727, 3483739,
    3483740, 3483741, 3483742, 3484003, 3484004, 3484005, 3484006, 3484007,
    3484008, 3484009, 3484034, 3484035, 3484042, 3484043, 3484046, 3484049,
    3484050, 3484051, 3484052, 3484053, 3484055, 3484073, 3484074, 3484075,
    3484084, 3484086, 3484092, 3484094, 3484096, 3484097, 3484098, 3484537,
    3484544, 3484629, 3484635, 3484636, 3484685, 3484687, 3484692, 3484699,
    3484703, 3484709, 3484979, 3485188, 3485189, 3485190, 3485191, 3485196,
    3485197, 3485198, 3485471, 3485475, 3485496, 3485550, 3485565, 3485570,
    3485606, 3485625, 3485630, 3485661, 3485871, 3485974, 3485981, 3485997,
    3486009, 3486014, 3486017, 3486072, 3486077, 3486078, 3486080, 3486110,
    3486112, 3486114, 3486116, 3486117, 3486119, 3486180, 3486185, 3486191,
    3486193, 3486194, 3486195, 3486202, 3486216, 3486217, 3486220, 3486223,
    3486229, 3486230, 3486510, 3486511, 3486514, 3486544, 3486550, 3486646,
    3486648, 3486651, 3486728, 3486754, 3486785, 3487022, 3487094, 3487134,
    3487140, 3487180, 3487216, 3487217, 3487219, 3487220, 3487221, 3487222,
    3487225, 3487241, 3487242, 3487243, 3487246, 3487247, 3487249, 3487250,
    3487252, 3487253, 3487254, 3487258, 3487264, 3487267, 3487268, 3487269,
    3487664, 3487670, 3487688, 3487689, 3487779, 3487888, 3488033, 3488186,
    3488211, 3488220, 3488256, 3488294, 3488303, 3488311, 3488319, 3488325,
    3488327, 3488345, 3488635, 3488646, 3488683, 3488720, 3488725, 3488727,
    3488731, 3488741, 3488746, 3488750, 3488774, 3488778, 3488792, 3488793,
    3488812, 3488823, 3489065, 3489066, 3489067, 3489167, 3489186, 3489237,
    3489238, 3489248, 3489277, 3489280, 3489283, 3489285, 3489290, 3489318,
    3489325, 3489326, 3489327, 3489328, 3489329, 3489330, 3489364, 3489467,
    3489479, 3489490, 3489533, 3489535, 3489536, 3489537, 3489564, 3489753,
    3489754, 3489755, 3489756, 3489757, 3489758, 3489759, 3490027, 3490270,
    3490279, 3490307, 3490315, 3490328, 3490338, 3490342, 3490347, 3490349,
    3490351, 3490352, 3490364, 3490370, 3490372, 3490374, 3490375, 3490381,
    3490382, 3490384, 3490386, 3490387, 3490398, 3490399, 3490400, 3490424,
    3490466, 3490566, 3490749, 3490834, 3490835, 3490838, 3490843, 3490845,
    3490861, 3490871, 3490881, 3490883, 3490890, 3490892, 3491008, 3491013,
    3491023, 3491181, 3491192, 3491208, 3491211, 3491214, 3491216, 3491239,
    3491243, 3491245, 3491250, 3491264, 3491277, 3491280, 3491292, 3491305,
    3491350, 3491407, 3491420, 3491424, 3491428, 3491441, 3491449, 3491514,
    3491591, 3491761, 3491762, 3491763, 3491764, 3491767, 3491818, 3491825,
    3491830, 3491833, 3491840, 3491858, 3491865, 3491867, 3491868, 3491983,
    3491987, 3492218, 3492258, 3492287, 3492289, 3492294, 3492296, 3492297,
    3492298, 3492302, 3492318, 3492321, 3492334, 3492336, 3492338, 3492345,
    3492420, 3492721, 3492771, 3492806, 3492808, 3492811, 3492815, 3492874,
    3492875, 3492881, 3492904, 3492906, 3492917, 3492922, 3492927, 3492928,
    3492930, 3492935, 3492944, 3492946, 3492947, 3493012, 3493013, 3493014,
    3493015, 3493016, 3493017, 3493018, 3493180, 3493192, 3493201, 3493246,
    3493298, 3493318, 3493347, 3493367, 3493370, 3493397, 3493598, 3493600,
    3493632, 3493676, 3493727, 3493785, 3493795, 3493801, 3493803, 3493805,
    3493806, 3493973, 3493974, 3493975, 3493976, 3493977, 3493978, 3493979,
    3494178, 3494181, 3494208, 3494230, 3494232, 3494234, 3494239, 3494241,
    3494244, 3494247, 3494253, 3494261, 3494271, 3494272, 3494273, 3494274,
    3494278, 3494283, 3494287, 3494288, 3494289, 3494294, 3494296, 3494324,
    3494329, 3494373, 3494434, 3494436, 3494439, 3494440, 3494488, 3494493,
    3494519, 3494531, 3494534, 3494535, 3494558, 3494659, 3494703, 3494747,
    3494748, 3494749, 3494750, 3494754, 3494761, 3494762, 3494764, 3494765,
    3494766, 3494770, 3494771, 3494774, 3494775, 3494778, 3494779, 3495009,
    3495020, 3495027, 3495280, 3495281, 3495282, 3495283, 3495284, 3495285,
    3495286, 3495617, 3495620, 3495739, 3495744, 3495754, 3495758, 3495763,
    3495766, 3495781, 3495832, 3495847, 3495850, 3495866, 3495892, 3495893,
    3496008, 3496010, 3496018, 3496021, 3496022, 3496023, 3496029, 3496030,
    3496032, 3496034, 3496035, 3496039, 3496040, 3496268, 3496274, 3496300,
    3496320, 3496380, 3496448, 3496449, 3496450, 3496451, 3496452, 3496453,
    3496454, 3496727, 3496765, 3496784, 3496787, 3496814, 3496845, 3496924,
    3497139, 3497142, 3497184, 3497265, 3497365, 3497394, 3497397, 3497411,
    3497421, 3497426, 3497427, 3497429, 3497430, 3497434, 3497438, 3497442,
    3497447, 3497471, 3497557, 3497559, 3497560, 3497561, 3497577, 3497578,
    3497581, 3497797, 3497816, 3497818, 3497819, 3497867, 3497873, 3497988,
    3498031, 3498058, 3498281, 3498299, 3498321, 3498344, 3498355, 3498356,
    3498360, 3498365, 3498368, 3498370, 3498371, 3498408, 3498420, 3498421,
    3498424, 3498428, 3498432, 3498433, 3498436, 3498588, 3498611, 3498612,
    3498613, 3498614, 3498615, 3498616, 3498617, 3498759, 3498774, 3498795,
    3498813, 3498852, 3498878, 3498881, 3498897, 3498899, 3498901, 3498965,
    3498969, 3499304, 3499337, 3499346, 3499371, 3499374, 3499378, 3499380,
    3499381, 3499382, 3499383, 3499387, 3499394, 3499395, 3499396, 3499397,
    3499398, 3499399, 3499401, 3499413, 3499416, 3499417, 3499418, 3499481,
    3499591, 3499594, 3499595, 3499596, 3499597, 3499598, 3499600, 3499839,
    3499859, 3499883, 3499889, 3499933, 3499952, 3499975, 3499990, 3500003,
    3500044, 3500139, 3500148, 3500151, 3500157, 3500222, 3500308, 3500346,
    3500360, 3500363, 3500366, 3500367, 3500374, 3500376, 3500378, 3500388,
    3500391, 3500393, 3500404, 3500444, 3500464, 3500729, 3500977, 3500978,
    3500979, 3500981, 3500982, 3500983, 3500984, 3500985, 3500986, 3500987,
    3500988, 3501280, 3501285, 3501307, 3501320, 3501327, 3501330, 3501331,
    3501372, 3501485, 3501494, 3501495, 3501498, 3501529, 3501530, 3501533,
    3501535, 3501540, 3501550, 3501552, 3501557, 3501558, 3501561, 3501563,
    3501565, 3501566, 3501571, 3501675, 3501760, 3501787, 3501848, 3501875,
    3501961, 3502054, 3502059, 3502186, 3502187, 3502189, 3502190, 3502191,
    3502192, 3502193, 3502387, 3502400, 3502402, 3502410, 3502469, 3502491,
    3502497, 3502542, 3502543, 3502550, 3502556, 3502583, 3502596, 3502759,
    3502760, 3502830, 3503004, 3503017, 3503092, 3503094, 3503095, 3503096,
    3503097, 3503098, 3503105, 3503284, 3503299, 3503305, 3503310, 3503311,
    3503312, 3503326, 3503332, 3503337, 3503347, 3503352, 3503353, 3503355,
    3503359, 3503363, 3503367, 3503374, 3503393, 3503414, 3503420, 3503422,
    3503457, 3503502, 3503528, 3503899, 3503932, 3503955, 3503984, 3504125,
    3504127, 3504129, 3504130, 3504131, 3504132, 3504319, 3504321, 3504331,
    3504342, 3504357, 3504358, 3504360, 3504371, 3504377, 3504389, 3504390,
    3504403, 3504406, 3504410, 3504426, 3504448, 3504449, 3504451, 3504473,
    3504477, 3504479, 3504526, 3504570, 3504781, 3504784, 3504823, 3504835,
    3504996, 3505130, 3505141, 3505143, 3505144, 3505147, 3505149, 3505150,
    3505472, 3505497, 3505508, 3505512, 3505670, 3505671, 3505807, 3505956,
    3505980, 3506033, 3506035, 3506330, 3483452, 3486583, 3489187, 3490748,
    3491276, 3497814, 3500059, 3480790, 3481362, 3481514, 3481668, 3481679,
    3482076, 3483119, 3483427, 3483537, 3483582, 3484045, 3484048, 3484058,
    3484090, 3484481, 3485456, 3485960, 3486012, 3486211, 3486215, 3487265,
    3487669, 3487671, 3488893, 3489258, 3489493, 3489494, 3489495, 3490123,
    3490335, 3490354, 3490396, 3490732, 3491369, 3491401, 3491448, 3492275,
    3492324, 3492836, 3492887, 3492939, 3493207, 3494266, 3494528, 3494757,
    3494759, 3494768, 3494798, 3496015, 3496067, 3496068, 3496278, 3496314,
    3497334, 3497444, 3497456, 3497996, 3498271, 3498357, 3498364, 3499201,
    3499428, 3499480, 3499958, 3500344, 3500353, 3500446, 3501548, 3501562,
    3501783, 3501800, 3502492, 3503324, 3503350, 3503499, 3504367, 3504397,
    3504454, 3504826, 3482313, 3484083, 3484484, 3487239, 3489107, 3490775,
    3477546, 3480785, 3480820, 3481717, 3482379, 3482671, 3482672, 3482929,
    3482930, 3482931, 3482932, 3483481, 3484038, 3484069, 3484078, 3484089,
    3485965, 3487218, 3487238, 3488351, 3488616, 3488624, 3489331, 3489332,
    3489547, 3490166, 3490419, 3490820, 3490855, 3491182, 3491771, 3492349,
    3492630, 3492857, 3493784, 3493787, 3493828, 3494352, 3494354, 3494355,
    3494357, 3494358, 3494360, 3494362, 3494363, 3494560, 3494787, 3494793,
    3495889, 3496099, 3496229, 3496240, 3498345, 3498441, 3499010, 3499012,
    3499013, 3499014, 3499016, 3499206, 3499926, 3499969, 3500315, 3500453,
    3501549, 3501794, 3502560, 3503465, 3503913, 3504402, 3504792, 3505025,
    3494356, 3481475, 3481677, 3482577, 3482916, 3483152, 3483437, 3484068,
    3484079, 3484080, 3485464, 3485679, 3485979, 3487227, 3487460, 3488765,
    3488769, 3488770, 3488871, 3490295, 3490362, 3490814, 3490840, 3491542,
    3491783, 3492817, 3492851, 3493278, 3493778, 3493808, 3494311, 3494314,
    3495667, 3496014, 3496782, 3497173, 3497175, 3497360, 3497453, 3497454,
    3498290, 3498403, 3498443, 3498828, 3499225, 3499379, 3499384, 3500193,
    3501308, 3501850, 3502452, 3502552, 3502675, 3503952, 3504364, 3504409,
    3505999, 3490394, 3491450, 3496065, 3501798, 3480825, 3480829, 3481638,
    3481640, 3481642, 3481645, 3482186, 3482397, 3482636, 3482637, 3482638,
    3482639, 3483532, 3483546, 3483566, 3483567, 3483568, 3483570, 3484057,
    3484059, 3484081, 3485149, 3485151, 3485152, 3485153, 3485154, 3485685,
    3485989, 3486022, 3486084, 3486088, 3486089, 3486090, 3486094, 3486228,
    3487149, 3488057, 3488258, 3488260, 3488261, 3488262, 3489216, 3489217,
    3489293, 3489295, 3489296, 3489297, 3490390, 3490402, 3490460, 3490868,
    3490869, 3490884, 3490885, 3490886, 3490887, 3490888, 3491804, 3491805,
    3491808, 3491809, 3491811, 3491812, 3491821, 3491822, 3491869, 3491871,
    3491872, 3491873, 3491874, 3492861, 3492936, 3492940, 3492941, 3492942,
    3492943, 3492945, 3493811, 3493840, 3493843, 3493845, 3493847, 3494286,
    3494783, 3494788, 3494792, 3495840, 3496174, 3496249, 3496264, 3496346,
    3496350, 3496383, 3496384, 3496385, 3496386, 3496387, 3497488, 3497489,
    3497490, 3497491, 3497493, 3498349, 3498437, 3498465, 3498468, 3498469,
    3498472, 3498478, 3499390, 3499403, 3499786, 3499787, 3499788, 3499789,
    3499791, 3500326, 3500329, 3500497, 3500499, 3500500, 3500501, 3501542,
    3502092, 3502093, 3502094, 3502095, 3502457, 3503018, 3503019, 3503020,
    3503021, 3503023, 3503379, 3503460, 3503933, 3504030, 3504031, 3504032,
    3504033, 3504034, 3504379, 3505059, 3505061, 3505062, 3505063, 3505064,
    3480789, 3480807, 3481523, 3481681, 3482334, 3482678, 3483298, 3483409,
    3483430, 3484037, 3484061, 3484062, 3484076, 3485678, 3485973, 3485986,
    3485990, 3487652, 3487707, 3488633, 3488846, 3489060, 3489086, 3489583,
    3490329, 3490332, 3490764, 3490816, 3491785, 3491786, 3491787, 3492849,
    3493241, 3493771, 3493809, 3494449, 3494555, 3494799, 3495762, 3495765,
    3495839, 3496212, 3496221, 3496809, 3497455, 3498350, 3498906, 3499209,
    3499342, 3499344, 3500160, 3500456, 3500458, 3501333, 3501532, 3502504,
    3502555, 3503314, 3503315, 3503904, 3504396, 3504790, 3505948, 3481518,
    3481622, 3481634, 3481637, 3481639, 3481646, 3482665, 3482666, 3482667,
    3482668, 3482669, 3482953, 3483737, 3484010, 3484011, 3484012, 3484013,
    3484014, 3485192, 3485193, 3485201, 3485203, 3485204, 3486197, 3486198,
    3486199, 3486200, 3486201, 3486218, 3487226, 3487229, 3487231, 3487234,
    3487235, 3488334, 3488336, 3488338, 3488341, 3488344, 3489323, 3489538,
    3489541, 3489760, 3489762, 3489763, 3489764, 3489766, 3490565, 3491017,
    3491018, 3491019, 3491020, 3491021, 3491968, 3491974, 3491976, 3491980,
    3493019, 3493021, 3493022, 3493023, 3493025, 3493696, 3493980, 3493981,
    3493982, 3493983, 3493985, 3494255, 3494665, 3494668, 3495287, 3495288,
    3495289, 3495290, 3495291, 3496455, 3496456, 3496457, 3496458, 3496459,
    3497413, 3497558, 3497566, 3497569, 3497574, 3497579, 3498619, 3498620,
    3498622, 3498623, 3498625, 3499391, 3499601, 3499602, 3499603, 3499604,
    3499610, 3500974, 3500975, 3500989, 3500990, 3500991, 3500992, 3500993,
    3502194, 3502195, 3502196, 3502197, 3502198, 3503099, 3503100, 3503101,
    3503102, 3503103, 3503335, 3503928, 3504133, 3504134, 3504135, 3504137,
    3504138, 3504908, 3505152, 3505153, 3505154, 3505155, 3480798, 3481957,
    3482913, 3484060, 3485445, 3486019, 3487542, 3488785, 3489572, 3490798,
    3491734, 3492853, 3494180, 3494753, 3496005, 3497181, 3498305, 3499351,
    3500307, 3503283, 3503930, 3477583, 3477585, 3480793, 3480797, 3480823,
    3481465, 3481671, 3481672, 3482536, 3482674, 3483142, 3483407, 3483549,
    3483550, 3483553, 3483578, 3484032, 3484036, 3484047, 3484082, 3485466,
    3485952, 3485953, 3485954, 3485961, 3486115, 3486213, 3486231, 3487248,
    3488641, 3488743, 3488771, 3489320, 3489322, 3489333, 3489334, 3489335,
    3490183, 3490261, 3490389, 3490825, 3490836, 3491244, 3491248, 3491769,
    3491823, 3492213, 3492237, 3492825, 3492882, 3492931, 3493788, 3493798,
    3493802, 3494204, 3494231, 3494364, 3494366, 3494735, 3494760, 3494772,
    3496019, 3496020, 3496027, 3496056, 3496201, 3496259, 3497439, 3498014,
    3498016, 3498341, 3498354, 3498426, 3498430, 3499309, 3499310, 3499311,
    3499313, 3499314, 3499316, 3499317, 3499377, 3499386, 3499389, 3499393,
    3499846, 3500340, 3500365, 3500369, 3500383, 3501567, 3501569, 3501780,
    3502546, 3503304, 3503346, 3503358, 3504359, 3504370, 3504373, 3504391,
    3505429, 3480788, 3481675, 3482670, 3483555, 3484041, 3484085, 3486005,
    3487240, 3488630, 3489503, 3490305, 3490837, 3491861, 3492840, 3493781,
    3494496, 3494773, 3495891, 3497354, 3498361, 3499236, 3500231, 3501471,
    3501746, 3503340, 3504362, 3504804, 3480809, 3481180, 3481183, 3481190,
    3481196, 3481217, 3481220, 3481705, 3481723, 3481951, 3481954, 3481960,
    3481963, 3481971, 3481975, 3481976, 3481977, 3482451, 3482679, 3482680,
    3482681, 3482998, 3483009, 3483010, 3483011, 3483067, 3483431, 3483454,
    3483562, 3483564, 3483565, 3483715, 3483716, 3483717, 3483718, 3483719,
    3484066, 3484067, 3484072, 3484516, 3484543, 3484547, 3484583, 3485453,
    3485481, 3485491, 3485495, 3485497, 3485502, 3485519, 3485522, 3485536,
    3485543, 3486015, 3486177, 3486225, 3486226, 3486478, 3486479, 3486480,
    3486481, 3487223, 3487224, 3487233, 3487261, 3487262, 3487592, 3487601,
    3487612, 3487672, 3487673, 3487684, 3488157, 3488649, 3488652, 3488654,
    3488655, 3488657, 3488660, 3488798, 3488829, 3488847, 3489195, 3489241,
    3489244, 3489247, 3489498, 3489501, 3489509, 3489511, 3489580, 3489581,
    3489589, 3490172, 3490194, 3490204, 3490216, 3490233, 3490236, 3490740,
    3490769, 3490772, 3491205, 3491218, 3491219, 3491221, 3491222, 3491230,
    3491231, 3491232, 3491321, 3491322, 3491325, 3491338, 3491790, 3491862,
    3492216, 3492217, 3492246, 3492248, 3492249, 3492729, 3492816, 3492870,
    3492871, 3493193, 3493194, 3493196, 3493199, 3493203, 3493264, 3493299,
    3493304, 3493775, 3493779, 3493780, 3493791, 3493793, 3494163, 3494164,
    3494167, 3494225, 3494235, 3494293, 3494541, 3494562, 3494570, 3494751,
    3494752, 3494782, 3494967, 3494975, 3494977, 3494980, 3494986, 3495661,
    3495662, 3495663, 3495685, 3495686, 3495688, 3495849, 3496011, 3496013,
    3496070, 3496073, 3496759, 3496790, 3496791, 3496798, 3496801, 3496802,
    3496813, 3496821, 3497357, 3497363, 3497387, 3497761, 3497763, 3497765,
    3497803, 3498276, 3498283, 3498301, 3498439, 3498809, 3498833, 3498836,
    3498844, 3498869, 3498873, 3498877, 3498893, 3499219, 3499223, 3499230,
    3499234, 3499410, 3499812, 3499815, 3499816, 3499817, 3499841, 3499919,
    3499927, 3499950, 3499968, 3499970, 3500311, 3500324, 3500732, 3500733,
    3500735, 3500736, 3501288, 3501290, 3501293, 3501298, 3501314, 3501315,
    3501526, 3501528, 3501534, 3501743, 3501971, 3502052, 3502060, 3502405,
    3502406, 3502407, 3502408, 3502413, 3502414, 3502490, 3503288, 3503289,
    3503290, 3503292, 3503295, 3503319, 3503371, 3503372, 3503456, 3503902,
    3503915, 3503922, 3504337, 3504344, 3504346, 3504350, 3504351, 3504354,
    3504380, 3504417, 3504420, 3504429, 3504430, 3504571, 3504802, 3504810,
    3504816, 3504830, 3505405, 3505407, 3505409, 3505454, 3480802, 3480803,
    3480832, 3481174, 3481268, 3481719, 3481736, 3481978, 3481982, 3482000,
    3482017, 3482054, 3482059, 3482682, 3483021, 3483022, 3483032, 3483307,
    3483457, 3483460, 3483545, 3483547, 3483551, 3483731, 3483733, 3483734,
    3483744, 3483745, 3484063, 3484064, 3484065, 3484071, 3484617, 3485552,
    3485584, 3485588, 3485595, 3485601, 3485604, 3486030, 3486203, 3486205,
    3486210, 3486580, 3486620, 3486650, 3487237, 3487256, 3487257, 3487260,
    3487630, 3487648, 3487691, 3487700, 3488681, 3488682, 3488690, 3488694,
    3488758, 3488857, 3489201, 3489203, 3489239, 3489240, 3489249, 3489550,
    3489553, 3489554, 3489555, 3489557, 3489585, 3489586, 3490325, 3490339,
    3490346, 3490567, 3490859, 3490872, 3491227, 3491353, 3491356, 3491373,
    3491777, 3491779, 3491781, 3492303, 3492308, 3492322, 3492703, 3492862,
    3492864, 3493198, 3493334, 3493350, 3493352, 3493374, 3494185, 3494186,
    3494187, 3494216, 3494218, 3494222, 3494256, 3494277, 3494699, 3494755,
    3494795, 3494797, 3495029, 3495032, 3495034, 3495040, 3495041, 3495042,
    3495716, 3495855, 3495894, 3496076, 3496800, 3496803, 3496807, 3496833,
    3496835, 3497372, 3497383, 3497424, 3497813, 3498358, 3498363, 3498854,
    3498859, 3498866, 3498879, 3498880, 3499375, 3499406, 3499475, 3499855,
    3499856, 3499858, 3499872, 3499874, 3500319, 3500321, 3500347, 3500406,
    3500725, 3500757, 3500758, 3500771, 3500773, 3501499, 3501500, 3501536,
    3501859, 3502397, 3502473, 3502480, 3502489, 3502544, 3502547, 3503364,
    3503366, 3503370, 3503373, 3503923, 3503925, 3504015, 3504353, 3504484,
    3504488, 3504540, 3504547, 3504819, 3505490, 3505492, 3505510, 3506016,
    3480795, 3481690, 3482676, 3483395, 3484031, 3485493, 3485993, 3487245,
    3488638, 3489223, 3489587, 3490870, 3491788, 3492711, 3493792, 3494184,
    3494756, 3495887, 3497352, 3498296, 3499370, 3500334, 3501525, 3502558,
    3504333, 3504832, 3480787, 3480791, 3481623, 3481669, 3481670, 3481684,
    3482684, 3483153, 3483573, 3484033, 3484040, 3484091, 3485469, 3485654,
    3485967, 3485975, 3486001, 3486111, 3486212, 3487228, 3487255, 3488775,
    3489104, 3489319, 3490312, 3490336, 3490831, 3491824, 3492327, 3492824,
    3492938, 3493804, 3494199, 3494758, 3494777, 3495879, 3497428, 3497747,
    3498003, 3498425, 3498435, 3499376, 3499388, 3500355, 3500357, 3501541,
    3501559, 3503348, 3504213, 3504355, 3504372, 3481387, 3482629, 3483425,
    3484611, 3486214, 3486232, 3487251, 3487266, 3489260, 3490392, 3491515,
    3492307, 3492814, 3493210, 3494259, 3494281, 3494780, 3496317, 3496321,
    3497795, 3498423, 3498447, 3499967, 3500440, 3501869, 3503360, 3504356,
    3504368, 3235351, 3262567
]

con = sqlite3.connect(DB_PATH)

# Usar IN com placeholders
placeholders = ",".join("?" for _ in codigos)
codigos_str = [str(c) for c in codigos]

cursor = con.execute(
    f'SELECT CODEUT FROM "inform_27_2026" WHERE CODEUT IN ({placeholders})',
    codigos_str
)

encontrados = set(row[0] for row in cursor.fetchall())
nao_encontrados = [c for c in codigos if str(c) not in encontrados]

con.close()

print(f"Total a validar: {len(codigos)}")
print(f"Encontrados na BD: {len(encontrados)}")
print(f"Não encontrados: {len(nao_encontrados)}")

if nao_encontrados:
    print("\nCódigos em falta:")
    for c in nao_encontrados:
        print(f"  {c}")